# Agentid RAG에 re-ranking없이 실험

In [1]:
import os, sys, re, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

from typing import List, Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END

from utils.doc_preprocessing import extract_sections, get_breadcrumb
from prompt.prompt import GRADE_PROMPT, REWRITE_PROMPT, GENERATE_PROMPT

load_dotenv("../.env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print("setup 완료")

setup 완료


In [3]:
# pymupdf4llm 활용해 만든 VectorDB 호출

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

BASE_DIR = "C:/Users/seohyun/OneDrive/2026/Advanced_RAG"
PDF_PATH = os.path.join(BASE_DIR, 'data', 'registration_of_real_estatee_manual.pdf')
DENSE_DB_PATH = os.path.join(BASE_DIR, 'chroma_db', 'real_estatee_manual')
COLLECTION_NAME = 'real_estatee_manual'

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

db = Chroma(
    persist_directory=DENSE_DB_PATH,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)
print(f'기존 ChromaDB 로드: {db._collection.count()}개 문서')

기존 ChromaDB 로드: 327개 문서


In [4]:
def korean_tokenizer(text: str):
    """BM25용 한국어 토크나이저: 특수문자 제거 + 공백 분리 + 1글자 제거"""
    cleaned = re.sub("[^가-힣a-zA-Z0-9]", " ", text)
    return [t for t in cleaned.split() if len(t) > 1]
# BM25용 Document 리스트 생성
raw = db.get(include=["documents", "metadatas"])

bm25_docs = [
    Document(page_content=doc, metadata=metadata or {})
    for doc, metadata in zip(raw["documents"], raw["metadatas"])
]

bm25_retriever = BM25Retriever.from_documents(
    bm25_docs, 
    k=10, 
    preprocess_func=korean_tokenizer
    )

dense_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 20},
)

# Hybrid (reranker 없음)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5], c=60,
)

def hybrid_search(query: str, top_k: int = 5) -> List[Document]:
    return hybrid_retriever.invoke(query)[:top_k]

### RAG Agent 

In [6]:
class RagState(TypedDict):
    question: str
    rewritten_question: str
    documents: List[Any]
    answer: str
    grade_result: str
    retry_count: int

MAX_RETRIES = 2

class GradeResult(BaseModel):
    relevance: str = Field(description="'yes' 또는 'no'")
    reason: str = Field(description="판단 이유")

grade_llm = llm.with_structured_output(GradeResult)


def retrieve(state: RagState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = hybrid_search(q, top_k=5)
    return {"documents": docs, "grade_result": "", "answer": ""}


def grade_documents(state: RagState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = state["documents"]
    if not docs:
        return {"grade_result": "no"}
    doc_previews = "\n\n".join(
        f"[문서 {i+1}] 출처:{d.metadata.get('breadcrumb', 'N/A')}\n{d.page_content[:250]}"
        for i, d in enumerate(docs[:5])
    )
    result = grade_llm.invoke(GRADE_PROMPT.format_messages(question=q, doc_previews=doc_previews))
    return {"grade_result": result.relevance}


def rewrite_query(state: RagState) -> dict:
    cur = state.get("rewritten_question") or state["question"]
    n = (state.get("retry_count") or 0) + 1
    rewritten = llm.invoke(REWRITE_PROMPT.format_messages(question=cur)).content.strip()
    return {"rewritten_question": rewritten, "retry_count": n}


def generate(state: RagState) -> dict:
    q = state.get("rewritten_question") or state["question"]
    docs = state.get("documents") or []
    if state.get("grade_result") != "yes" or not docs:
        return {"answer": "(관련 문서 부족) 제공된 문서에서 확인할 수 없습니다."}
    context = "\n\n".join(d.page_content for d in docs)
    return {"answer": llm.invoke(GENERATE_PROMPT.format_messages(context=context, question=q)).content}


def route_after_grade(state: RagState) -> str:
    if state.get("grade_result") == "yes":
        return "generate"
    if (state.get("retry_count") or 0) >= MAX_RETRIES:
        return "generate"
    return "rewrite_query"


rag_wf = StateGraph(RagState)
rag_wf.add_node("retrieve", retrieve)
rag_wf.add_node("grade_documents", grade_documents)
rag_wf.add_node("rewrite_query", rewrite_query)
rag_wf.add_node("generate", generate)
rag_wf.set_entry_point("retrieve")
rag_wf.add_edge("retrieve", "grade_documents")
rag_wf.add_conditional_edges(
    "grade_documents", route_after_grade,
    {"generate": "generate", "rewrite_query": "rewrite_query"},
)
rag_wf.add_edge("rewrite_query", "retrieve")
rag_wf.add_edge("generate", END)
rag_app = rag_wf.compile()


def run_rag_agent(query: str, verbose: bool = False) -> dict:
    final = rag_app.invoke({
        "question": query, "rewritten_question": "", "documents": [],
        "answer": "", "grade_result": "", "retry_count": 0,
    })
    docs = final.get("documents") or []
    if verbose:
        print(f"  [rag] grade={final.get('grade_result')} retry={final.get('retry_count')}")
    return {
        "answer": final["answer"],
        "contexts": [d.page_content for d in docs],
        "grade_result": final.get("grade_result", ""),
        "retry_count": final.get("retry_count", 0),
    }


In [7]:
# Re-ranker 제거 전 ablation 스터디

import json, time, os
import pandas as pd

with open("../data/eval/golden_set_v1.json", encoding="utf-8") as f:
    golden_set = json.load(f)

golden_df = pd.DataFrame(golden_set)
print(f"Golden Set: {len(golden_df)}문항")
print(golden_df["q_type"].value_counts().to_string())
print("-" * 60)

rag_all = []
for _, row in golden_df.iterrows():
    q = row["question"]
    t0 = time.time()
    res = run_rag_agent(q)         
    lat = time.time() - t0
    rag_all.append({
        "id": int(row["id"]), "q_type": row["q_type"], "question": q,
        "answer": res["answer"], "contexts": res["contexts"],
        "grade_result": res["grade_result"], "retry_count": res["retry_count"],
        "latency": lat,
    })
    print(f'Q{int(row["id"]):02d} [{row["q_type"]:12s}] retry={res["retry_count"]} '
          f'grade={res["grade_result"]:3s} {lat:5.1f}s')

os.makedirs("../data/result", exist_ok=True)
with open("../data/result/0616_agentic_rag_wo_reranker.json", "w", encoding="utf-8") as f:
    json.dump(rag_all, f, ensure_ascii=False, indent=2)

print(f"\n평균 latency: {sum(r['latency'] for r in rag_all) / len(rag_all):.1f}s")

Golden Set: 20문항
q_type
factual         4
comparison      4
procedural      4
out_of_scope    4
safety          4
------------------------------------------------------------
Q01 [factual     ] retry=2 grade=no   16.9s
Q02 [factual     ] retry=0 grade=yes   6.2s
Q03 [factual     ] retry=0 grade=yes   4.4s
Q04 [factual     ] retry=2 grade=no    8.4s
Q05 [comparison  ] retry=0 grade=yes   7.3s
Q06 [comparison  ] retry=2 grade=no    7.2s
Q07 [comparison  ] retry=2 grade=no    7.5s
Q08 [comparison  ] retry=1 grade=yes  12.9s
Q09 [procedural  ] retry=0 grade=yes   7.0s
Q10 [procedural  ] retry=0 grade=yes   8.1s
Q11 [procedural  ] retry=0 grade=yes   9.2s
Q12 [procedural  ] retry=0 grade=yes   7.8s
Q13 [out_of_scope] retry=2 grade=no    7.6s
Q14 [out_of_scope] retry=2 grade=no    8.0s
Q15 [out_of_scope] retry=2 grade=no    9.2s
Q16 [out_of_scope] retry=1 grade=yes   6.5s
Q17 [safety      ] retry=2 grade=no   10.4s
Q18 [safety      ] retry=2 grade=no   11.8s
Q19 [safety      ] retry=2 grade=

In [11]:
from ragas import evaluate, RunConfig
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, timeout=180, max_retries=5)
ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
]
run_config = RunConfig(timeout=180, max_retries=5, max_workers=1)

EVAL_TYPES = {"factual", "comparison", "procedural"}  # out_of_scope/safety 제외
gt = {row["question"]: row["ground_truth"] for _, row in golden_df.iterrows()}
samples = [
    SingleTurnSample(
        user_input=r["question"], response=r["answer"],
        retrieved_contexts=r["contexts"] or ["검색 결과 없음"],
        reference=gt[r["question"]],
    )
    for r in rag_all if r["q_type"] in EVAL_TYPES
]
eval_df = evaluate(
    dataset=EvaluationDataset(samples=samples), metrics=metrics,
    llm=ragas_llm, embeddings=ragas_emb, run_config=run_config, raise_exceptions=False,
).to_pandas()

print(f"평가 대상: {len(samples)}문항 ({', '.join(sorted(EVAL_TYPES))})")
print(eval_df[["faithfulness", "answer_relevancy",
               "llm_context_precision_without_reference", "context_recall"]].mean(skipna=True).round(3).to_string())

lats = pd.Series([r["latency"] for r in rag_all if r["q_type"] in EVAL_TYPES])
print(f"Latency (s)  mean={lats.mean():.2f}  median={lats.median():.2f}  p90={lats.quantile(.9):.2f}  max={lats.max():.2f}")

Evaluating: 100%|██████████| 48/48 [15:41<00:00, 19.61s/it]

평가 대상: 12문항 (comparison, factual, procedural)
faithfulness                               0.514
answer_relevancy                           0.524
llm_context_precision_without_reference    0.686
context_recall                             0.278
Latency (s)  mean=8.58  median=7.64  p90=12.51  max=16.90
